# LeetCode 3360 — Stone Removal Game bằng Python

Link đề: https://leetcode.com/problems/stone-removal-game/ · Độ khó: **Easy** · Chủ đề: Math, Simulation (mô phỏng), Game (trò chơi)

Skeleton LeetCode:

```python
class Solution:
    def canAliceWin(self, n: int) -> bool:
        ...
```

Ghi chú: mọi đáp án trong notebook đã được kiểm chứng bằng code (kể cả vét cạn **toàn bộ** `n = 1..50` và đối chiếu năm cách làm với nhau). Hai ví dụ chính lấy từ đề; ví dụ tự dựng thêm có ghi rõ "tự dựng" — nếu cần khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: Alice và Bob lần lượt bốc đá từ một đống `n` viên; Alice đi trước và phải bốc **đúng `10`** viên; mỗi lượt sau, người chơi phải bốc **ít hơn lượt của đối thủ đúng `1` viên** (`10, 9, 8, ...`); ai **không bốc được** thì thua. Trả `True` nếu Alice thắng.
2. Nắm các công cụ Python: vòng `while` với biến đếm, **tính chẵn/lẻ** bằng `% 2`, đổi lượt bằng `turn = 1 - turn`, tổng tích luỹ `itertools.accumulate`, tìm kiếm nhị phân `bisect`, đệ quy có nhớ `functools.cache`, và `range` đếm lùi.
3. Đi từ cách **mô phỏng từng lượt có tên người chơi** → **đếm số lượt `k` rồi xét chẵn/lẻ** → **đệ quy trò chơi** `win(n, x)` → **tổng tích luỹ + `bisect`** → **bảng khoảng thắng tính sẵn**.
4. Hiểu mấu chốt: trò chơi này **không có lựa chọn** (mỗi lượt số viên phải bốc là **cố định**), nên chỉ là một **mô phỏng**. Gọi `k` là số lượt thực hiện được; người thực hiện lượt cuối cùng thắng (người kia không bốc được); Alice đi lượt lẻ nên **Alice thắng ⇔ `k` lẻ**.
5. Tránh các lỗi kinh điển: dùng `n > x` thay vì `n >= x` (không cho bốc hết đống), đảo nhầm điều kiện thắng (`k` chẵn), nhầm rằng "ai bốc nhiều hơn thì thắng", và để vòng lặp chạy khi `x` chạm `0` (chỉ an toàn nhờ ràng buộc `n <= 50`).

## 2. Đề bài & ví dụ

Alice và Bob chơi một trò chơi lần lượt bốc đá từ một đống, **Alice đi trước**.

- Lượt đầu, Alice bốc **đúng `10`** viên đá.
- Ở mỗi lượt sau, mỗi người chơi bốc **đúng ít hơn `1` viên** so với lượt trước của đối thủ.

Người **không thể thực hiện lượt đi** (đống không đủ số viên phải bốc) **thua** trò chơi.

Cho số nguyên dương `n` (số viên đá ban đầu). Trả về `true` nếu Alice thắng, ngược lại `false`.

**Ví dụ 1**

- Input: `n = 12`
- Output: `true`
- Giải thích: Alice bốc `10` viên, còn `2` viên cho Bob. Bob không thể bốc `9` viên nên Alice thắng.

**Ví dụ 2**

- Input: `n = 1`
- Output: `false`
- Giải thích: Alice không thể bốc `10` viên nên Alice thua.

**Ràng buộc**

- `1 <= n <= 50`

In [1]:
examples = [
    (12, True),
    (1, False),
]

def can_alice_win_quick(n: int) -> bool:
    """Bản nhanh để chạy thử ví dụ: mô phỏng, đếm số lượt."""
    x, k = 10, 0
    while n >= x:
        n -= x
        x -= 1
        k += 1
    return k % 2 == 1

for n, expected in examples:
    print(f"n={n} -> {can_alice_win_quick(n)} (kỳ vọng {expected})")
    assert can_alice_win_quick(n) == expected

n=12 -> True (kỳ vọng True)
n=1 -> False (kỳ vọng False)


## 3. Kiến thức nền

### 3.1. Vòng `while` và biến đếm

`while điều_kiện:` lặp **chừng nào điều kiện còn đúng**. Ở bài này, điều kiện là "đống còn đủ viên cho lượt này": `n >= x` (với `x` là số viên phải bốc). Mỗi lần lặp ta trừ `x` khỏi `n`, giảm `x` đi `1`, và tăng biến đếm `k` (số lượt đã đi).

In [2]:
n, x, k = 27, 10, 0
while n >= x:
    print(f"lượt {k + 1}: còn n={n}, phải bốc x={x}")
    n -= x
    x -= 1
    k += 1
print("hết lượt đi được: n =", n, ", x =", x, ", k =", k)

lượt 1: còn n=27, phải bốc x=10
lượt 2: còn n=17, phải bốc x=9
lượt 3: còn n=8, phải bốc x=8
hết lượt đi được: n = 0 , x = 7 , k = 3


### 3.2. Chẵn/lẻ bằng `% 2` và vì sao `k` lẻ nghĩa là Alice thắng

`k % 2` là phần dư khi chia `k` cho `2`: bằng `1` nếu `k` lẻ, `0` nếu `k` chẵn. Các lượt được đánh số `1, 2, 3, ...`: Alice đi các lượt **lẻ** (1, 3, 5, ...), Bob đi các lượt **chẵn**. Nếu tổng số lượt thực hiện được là `k`, thì **người đi lượt thứ `k` là người đi cuối**, và người kế tiếp (lượt `k + 1`) không đi được ⇒ thua. Vậy: `k` lẻ ⇒ Alice đi cuối ⇒ **Alice thắng**.

In [3]:
for k in range(0, 6):
    nguoi_cuoi = "không ai (Alice thua ngay)" if k == 0 else ("Alice" if k % 2 == 1 else "Bob")
    nguoi_thang = "Alice" if k % 2 == 1 else "Bob"
    print(f"k={k}: k % 2 = {k % 2}, người đi cuối = {nguoi_cuoi}, người thắng = {nguoi_thang}")

k=0: k % 2 = 0, người đi cuối = không ai (Alice thua ngay), người thắng = Bob
k=1: k % 2 = 1, người đi cuối = Alice, người thắng = Alice
k=2: k % 2 = 0, người đi cuối = Bob, người thắng = Bob
k=3: k % 2 = 1, người đi cuối = Alice, người thắng = Alice
k=4: k % 2 = 0, người đi cuối = Bob, người thắng = Bob
k=5: k % 2 = 1, người đi cuối = Alice, người thắng = Alice


### 3.3. Đổi lượt bằng `turn = 1 - turn`

Một cách khác để biết ai đang đi: biến `turn` nhận `0` (Alice) hoặc `1` (Bob). Đổi lượt: `turn = 1 - turn` (nếu `turn = 0` thì thành `1`, nếu `1` thì thành `0`). Dùng danh sách tên `["Alice", "Bob"][turn]` để in ra.

In [4]:
names = ["Alice", "Bob"]
turn = 0
for step in range(5):
    print(f"lượt {step + 1}: {names[turn]}")
    turn = 1 - turn

lượt 1: Alice
lượt 2: Bob
lượt 3: Alice
lượt 4: Bob
lượt 5: Alice


### 3.4. Tổng tích luỹ: `itertools.accumulate`

Số viên đã bốc sau `k` lượt là `10 + 9 + ... + (11 - k)`. `accumulate([10, 9, 8, ...])` sinh ra các **tổng dồn**: `10, 19, 27, ...` — đây là số viên **tối thiểu** cần có để đi đủ `1, 2, 3, ...` lượt.

In [5]:
from itertools import accumulate

moves = list(range(10, 0, -1))              # [10, 9, 8, ..., 1]
print(moves)
prefix = list(accumulate(moves))
print(prefix)                               # [10, 19, 27, 34, 40, 45, 49, 52, 54, 55]
print(sum(moves))                           # 55 = tổng cả 10 lượt

[10, 9, 8, 7, 6, 5, 4, 3, 2, 1]
[10, 19, 27, 34, 40, 45, 49, 52, 54, 55]
55


### 3.5. `bisect_right`: đếm bao nhiêu phần tử `<= n`

Cho danh sách **đã sắp xếp** `a`, `bisect_right(a, n)` trả về **số phần tử `<= n`** (vị trí chèn bên phải). Ở đây số lượt `k` đi được đúng bằng số tổng dồn `<= n`: `k = bisect_right(prefix, n)`.

In [6]:
from bisect import bisect_right

prefix = [10, 19, 27, 34, 40, 45, 49, 52, 54, 55]
for n in (9, 10, 18, 19, 26, 27, 50):
    print(f"n={n}: k = {bisect_right(prefix, n)}")

n=9: k = 0
n=10: k = 1
n=18: k = 1
n=19: k = 2
n=26: k = 2
n=27: k = 3
n=50: k = 7


### 3.6. Đệ quy có nhớ: `functools.cache`

Một hàm **đệ quy** tự gọi lại chính nó với bài toán nhỏ hơn. `@cache` ghi nhớ kết quả đã tính để khỏi tính lại. Ta sẽ dùng nó ở Cách 3 để biểu diễn "người sắp đi thắng khi nào".

In [7]:
from functools import cache

@cache
def tong_den(n):
    return 0 if n == 0 else n + tong_den(n - 1)

print(tong_den(10))        # 55
print(tong_den(4))         # 10

55
10


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `n = 33`:

| lượt | người đi | phải bốc | còn lại trước | còn lại sau |
|---|---|---|---|---|
| 1 | Alice | 10 | 33 | 23 |
| 2 | Bob | 9 | 23 | 14 |
| 3 | Alice | 8 | 14 | 6 |
| 4 | Bob | 7 | 6 | **không bốc được** (`6 < 7`) |

Bob không đi được ⇒ Bob thua ⇒ **Alice thắng**. Số lượt đi được `k = 3` (lẻ). Với `n = 19`: Alice `10` (còn `9`), Bob `9` (còn `0`), Alice phải bốc `8` nhưng còn `0` ⇒ Alice thua; `k = 2` (chẵn).

**Điều đáng chú ý:** trò chơi **không có lựa chọn nào** — mỗi lượt số viên phải bốc đã cố định (`10, 9, 8, ...`). Vì vậy ta không cần "tính chiến thuật tối ưu", chỉ cần **mô phỏng** và xem ai đi lượt cuối.

In [8]:
def show_game(n):
    x, k = 10, 0
    names = ["Alice", "Bob"]
    print(f"n = {n}")
    while n >= x:
        print(f"  {names[k % 2]} bốc {x}: {n} -> {n - x}")
        n -= x
        x -= 1
        k += 1
    loser = names[k % 2]
    print(f"  {loser} cần bốc {x} nhưng chỉ còn {n} ⇒ {loser} thua. k = {k}")

show_game(33)
show_game(19)
show_game(9)

n = 33
  Alice bốc 10: 33 -> 23
  Bob bốc 9: 23 -> 14
  Alice bốc 8: 14 -> 6
  Bob cần bốc 7 nhưng chỉ còn 6 ⇒ Bob thua. k = 3
n = 19
  Alice bốc 10: 19 -> 9
  Bob bốc 9: 9 -> 0
  Alice cần bốc 8 nhưng chỉ còn 0 ⇒ Alice thua. k = 2
n = 9
  Alice cần bốc 10 nhưng chỉ còn 9 ⇒ Alice thua. k = 0


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng lượt có tên người chơi

**Ý tưởng.** Làm đúng như đề: một vòng `while`; ở mỗi lượt kiểm tra "người đang đi có đủ viên không?". Nếu **không đủ** thì **người này thua**, trả kết quả luôn (Alice thua nếu người đó là Alice). Nếu đủ thì bốc, giảm `x`, đổi lượt.

In [9]:
def can_alice_win_turns(n: int) -> bool:
    x = 10                 # số viên phải bốc ở lượt hiện tại
    turn = 0               # 0: Alice, 1: Bob
    while True:
        if n < x:          # không bốc được -> người đang đi thua
            return turn == 1          # nếu Bob thua thì Alice thắng
        n -= x
        x -= 1
        turn = 1 - turn

print(can_alice_win_turns(12))    # True
print(can_alice_win_turns(1))     # False
print(can_alice_win_turns(33))    # True
print(can_alice_win_turns(19))    # False

True
False
True
False


**Độ phức tạp (Cách 1).** Mỗi lượt `x` giảm `1`, và tổng số viên cần là `10 + 9 + ...`; số lượt tối đa `O(sqrt(n))` (vì `1 + 2 + ... + m ≈ m²/2`), thực tế `<= 10` với `n <= 50`. **Time `O(sqrt(n))`** (hằng số với ràng buộc), **Space `O(1)`**.

**Nút thắt.** Không có nút thắt đáng kể — trò chơi nhỏ. Ta chỉ muốn **gọn hơn** (chỉ đếm số lượt `k`, bỏ biến `turn`) và hiểu thêm các cách nhìn khác (đệ quy, tổng tích luỹ, bảng tra).

**Cẩn thận vòng lặp.** `while True` kết thúc nhờ `n < x` cuối cùng xảy ra. Điều này đúng vì `n <= 50 < 55 = 10 + 9 + ... + 1`, nên **không bao giờ** đến lượt phải bốc `0` viên. Ngoài ràng buộc (n >= 55) cần thêm điều kiện dừng (xem mục 10 và 13).

## 6. Ý tưởng thuật toán / bước đột phá

**Bỏ biến `turn`, chỉ đếm số lượt `k`.** Ai đi lượt cuối là người **thắng**. Lượt thứ `i` (đếm từ `1`) do Alice đi nếu `i` lẻ. Vậy:

> Alice thắng ⇔ số lượt đã thực hiện `k` là **số lẻ**.

Thật vậy, sau `k` lượt đi được, lượt `k + 1` không đi được; người đi lượt `k + 1` thua. Lượt `k + 1` là của Bob khi `k + 1` chẵn, tức `k` lẻ ⇒ Bob thua, Alice thắng.

Đặc biệt `n < 10` ⇒ `k = 0` (chẵn) ⇒ Alice thua ngay (đúng ví dụ 2).

In [10]:
def count_moves(n: int) -> int:
    x, k = 10, 0
    while n >= x:
        n -= x
        x -= 1
        k += 1
    return k

for n in (1, 9, 10, 12, 18, 19, 26, 27, 33, 34, 50):
    k = count_moves(n)
    print(f"n={n:>2}: k={k}, k lẻ? {k % 2 == 1}")

n= 1: k=0, k lẻ? False
n= 9: k=0, k lẻ? False
n=10: k=1, k lẻ? True
n=12: k=1, k lẻ? True
n=18: k=1, k lẻ? True
n=19: k=2, k lẻ? False
n=26: k=2, k lẻ? False
n=27: k=3, k lẻ? True
n=33: k=3, k lẻ? True
n=34: k=4, k lẻ? False
n=50: k=7, k lẻ? True


## 7. Các cách tiếp theo

### Cách 2 — Đếm số lượt `k` rồi xét chẵn/lẻ (lời giải chính thức)

**Giải thích từng đoạn.**

1. `x, k = 10, 0` — `x` là số viên phải bốc ở lượt này, `k` là số lượt đã đi.
2. `while n >= x:` — còn đủ viên cho lượt này thì cứ đi.
3. `n -= x; x -= 1; k += 1` — bốc `x` viên, lượt sau bốc ít hơn `1`, ghi nhận `1` lượt.
4. `return k % 2 == 1` — Alice thắng khi `k` lẻ.

In [11]:
def can_alice_win_count(n: int) -> bool:
    x, k = 10, 0
    while n >= x:
        n -= x
        x -= 1
        k += 1
    return k % 2 == 1

print(can_alice_win_count(12))    # True
print(can_alice_win_count(1))     # False
print(can_alice_win_count(33))    # True

True
False
True


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionCount:
    def canAliceWin(self, n: int) -> bool:
        x, k = 10, 0
        while n >= x:
            n -= x
            x -= 1
            k += 1
        return k % 2 == 1

print(SolutionCount().canAliceWin(12))   # True

True


**Độ phức tạp Cách 2:** **Time `O(sqrt(n))`** (hằng số: tối đa `10` lượt), **Space `O(1)`**.

### Cách 3 — Đệ quy trò chơi: `win(n, x)`

**Giải thích.** Gọi `win(n, x)` = "người **sắp đi**, phải bốc `x` viên, đống còn `n` viên, thì **thắng** hay không?". Nếu `n < x`: người này **không đi được** ⇒ **thua**. Ngược lại người này bốc `x`, đối thủ phải bốc `x - 1` với đống `n - x`; người này thắng **khi và chỉ khi đối thủ thua**: `win(n, x) = not win(n - x, x - 1)`. Alice thắng ⇔ `win(n, 10)`.

Đây là cách giải **trò chơi** nói chung (cho dù có lựa chọn hay không); bài này không có lựa chọn nên đệ quy chỉ là một "chuỗi" nối tiếp.

In [13]:
from functools import cache

@cache
def win(n: int, x: int) -> bool:
    """Người sắp đi phải bốc x viên, đống có n viên. Trả True nếu người đó thắng."""
    if n < x:
        return False                 # không bốc được -> thua
    return not win(n - x, x - 1)     # thắng nếu đối thủ (bốc x-1, đống n-x) thua

def can_alice_win_recursive(n: int) -> bool:
    return win(n, 10)

print(can_alice_win_recursive(12))    # True
print(can_alice_win_recursive(1))     # False
print(can_alice_win_recursive(19))    # False
print(win(2, 9), "(Bob không bốc được 9 viên từ 2)")

True
False
False
False (Bob không bốc được 9 viên từ 2)


In [14]:
# Code nộp LeetCode cho Cách 3
from functools import cache

class SolutionRecursive:
    def canAliceWin(self, n: int) -> bool:
        @cache
        def win(m: int, x: int) -> bool:
            if m < x:
                return False
            return not win(m - x, x - 1)
        return win(n, 10)

print(SolutionRecursive().canAliceWin(33))   # True

True


**Độ phức tạp Cách 3:** **Time `O(sqrt(n))`** (mỗi lượt một lời gọi, tối đa `10`), **Space `O(sqrt(n))`** cho ngăn xếp đệ quy và bộ nhớ đệm (hằng số ở đây).

> Lưu ý: `win(n, x)` chỉ kết thúc vì `x` giảm và `n` giảm. Với `n <= 50` ta không bao giờ đến `x = 0` (vì `10 + ... + 1 = 55 > 50`).

### Cách 4 — Tổng tích luỹ + `bisect_right`

**Giải thích.** Số viên cần để đi trọn `m` lượt đầu là `10 + 9 + ... + (11 - m)`. Dựng `prefix = list(accumulate(range(10, 0, -1)))` một lần. Số lượt đi được là `k = bisect_right(prefix, n)` (số mốc `<= n`). Rồi xét `k` lẻ. Cách này **không cần vòng lặp trừ dần**; hợp khi trả lời nhiều truy vấn `n`.

In [15]:
from itertools import accumulate
from bisect import bisect_right

PREFIX = list(accumulate(range(10, 0, -1)))     # [10, 19, 27, 34, 40, 45, 49, 52, 54, 55]

def can_alice_win_bisect(n: int) -> bool:
    k = bisect_right(PREFIX, n)
    return k % 2 == 1

print(PREFIX)
print(can_alice_win_bisect(12))    # True
print(can_alice_win_bisect(1))     # False
print(can_alice_win_bisect(27))    # True (k = 3)

[10, 19, 27, 34, 40, 45, 49, 52, 54, 55]
True
False
True


In [16]:
# Code nộp LeetCode cho Cách 4
from itertools import accumulate
from bisect import bisect_right

class SolutionBisect:
    def canAliceWin(self, n: int) -> bool:
        prefix = list(accumulate(range(10, 0, -1)))
        return bisect_right(prefix, n) % 2 == 1

print(SolutionBisect().canAliceWin(27))   # True

True


**Độ phức tạp Cách 4:** **Time `O(log m)`** cho mỗi truy vấn với `m = 10` mốc (tức hằng số), dựng bảng `O(m)`; **Space `O(m)`**.

### Cách 5 — Bảng khoảng thắng tính sẵn

**Giải thích.** Với `n <= 50`, các khoảng `n` mà Alice thắng là những đoạn nằm giữa mốc thứ `k` và mốc thứ `k + 1` với `k` **lẻ**: `[10, 18]`, `[27, 33]`, `[40, 44]`, `[49, 50]` (ghi theo `prefix = 10, 19, 27, 34, 40, 45, 49, ...`). Ta có thể ghi **cứng** thành một hàm kiểm tra khoảng. Cách này nhanh và "ngắn" nhưng **dễ sai nếu tính khoảng bằng tay** — nên luôn **kiểm chứng bằng mô phỏng** (như bên dưới).

In [17]:
WIN_RANGES = [(10, 18), (27, 33), (40, 44), (49, 50)]

def can_alice_win_ranges(n: int) -> bool:
    return any(lo <= n <= hi for lo, hi in WIN_RANGES)

# Kiểm chứng bảng khoảng bằng mô phỏng, cho mọi n hợp lệ
for n in range(1, 51):
    assert can_alice_win_ranges(n) == can_alice_win_count(n), n
print("Bảng khoảng khớp mô phỏng với mọi n = 1..50")

winning = [n for n in range(1, 51) if can_alice_win_count(n)]
print(winning)

Bảng khoảng khớp mô phỏng với mọi n = 1..50
[10, 11, 12, 13, 14, 15, 16, 17, 18, 27, 28, 29, 30, 31, 32, 33, 40, 41, 42, 43, 44, 49, 50]


In [18]:
# Code nộp LeetCode cho Cách 5
class SolutionRanges:
    def canAliceWin(self, n: int) -> bool:
        return 10 <= n <= 18 or 27 <= n <= 33 or 40 <= n <= 44 or 49 <= n <= 50

print(SolutionRanges().canAliceWin(12))   # True
print(SolutionRanges().canAliceWin(19))   # False

True
False


**Độ phức tạp Cách 5:** **Time `O(1)`**, **Space `O(1)`**. (Chỉ đúng với `n <= 50`.)

## 8. TRACE từng bước

Ta trace Cách 2 trên ba input: `n = 33` (Alice thắng, `k = 3`), `n = 45` (Bob thắng, `k = 6`) và `n = 12` (ví dụ đề). Mỗi lượt in `n` trước khi bốc, `x`, `n` sau khi bốc, `k` và ai vừa đi.

In [19]:
def trace_game(n: int) -> bool:
    x, k = 10, 0
    names = ["Alice", "Bob"]
    print(f"n = {n}")
    while n >= x:
        who = names[k % 2]
        print(f"  lượt {k + 1} ({who}): n={n}, x={x} -> n sau = {n - x}")
        n -= x
        x -= 1
        k += 1
    loser = names[k % 2]
    print(f"  dừng: n={n} < x={x}; k={k} ({'lẻ' if k % 2 else 'chẵn'}) -> {loser} không đi được")
    result = k % 2 == 1
    print("  Alice thắng?", result)
    return result

trace_game(33)
print()
trace_game(45)
print()
trace_game(12)

n = 33
  lượt 1 (Alice): n=33, x=10 -> n sau = 23
  lượt 2 (Bob): n=23, x=9 -> n sau = 14
  lượt 3 (Alice): n=14, x=8 -> n sau = 6
  dừng: n=6 < x=7; k=3 (lẻ) -> Bob không đi được
  Alice thắng? True

n = 45
  lượt 1 (Alice): n=45, x=10 -> n sau = 35
  lượt 2 (Bob): n=35, x=9 -> n sau = 26
  lượt 3 (Alice): n=26, x=8 -> n sau = 18
  lượt 4 (Bob): n=18, x=7 -> n sau = 11
  lượt 5 (Alice): n=11, x=6 -> n sau = 5
  lượt 6 (Bob): n=5, x=5 -> n sau = 0
  dừng: n=0 < x=4; k=6 (chẵn) -> Alice không đi được
  Alice thắng? False

n = 12
  lượt 1 (Alice): n=12, x=10 -> n sau = 2
  dừng: n=2 < x=9; k=1 (lẻ) -> Bob không đi được
  Alice thắng? True


True

Bảng trace cho `n = 33`:

| lượt `k+1` | người đi | `x` | `n` trước | `n` sau | `n >= x`? |
|---|---|---|---|---|---|
| 1 | Alice | 10 | 33 | 23 | có |
| 2 | Bob | 9 | 23 | 14 | có |
| 3 | Alice | 8 | 14 | 6 | có |
| 4 | Bob | 7 | 6 | — | **không** (dừng) |

`k = 3` lẻ ⇒ Alice thắng. Với `n = 45`: `10, 9, 8, 7, 6, 5` lấy hết `45` ⇒ `k = 6` (Bob đi cuối), Alice cần bốc `4` nhưng còn `0` ⇒ Alice thua.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (đếm số lượt rồi xét chẵn/lẻ): ngắn, đúng và dễ hiểu.

In [20]:
class Solution:
    def canAliceWin(self, n: int) -> bool:
        x, k = 10, 0
        while n >= x:
            n -= x
            x -= 1
            k += 1
        return k % 2 == 1

sol = Solution()
print(sol.canAliceWin(12))    # True
print(sol.canAliceWin(1))     # False

True
False


## 10. Vì sao thuật toán đúng

- **Không có lựa chọn:** mỗi lượt, số viên phải bốc **bị quy định** (`10, 9, 8, ...`), nên toàn bộ trò chơi là một dãy xác định theo `n`. Không có "chiến thuật" để cân nhắc.
- **Dừng đúng lúc:** vòng lặp chạy khi đống còn **ít nhất `x`** viên; ngay khi `n < x`, người đến lượt không bốc được ⇒ thua, đúng như đề ("người không thể đi thì thua").
- **Ai đi lượt cuối?** Các lượt được đánh số `1, 2, ..., k`; Alice đi lượt lẻ. Người đi lượt `k` là người **cuối cùng bốc được**; đối thủ của họ (lượt `k + 1`) thua. Vậy người đi lượt `k` thắng; đó là Alice **iff** `k` lẻ.
- **Trường hợp `k = 0`** (`n < 10`): Alice không đi được lượt đầu ⇒ thua; `0` chẵn ⇒ trả `False` ✓.
- **Vì sao không vô hạn:** `x` giảm `1` mỗi lượt và tổng các lượt `10 + 9 + ... + 1 = 55 > 50 >= n`. Nên với mọi `n` hợp lệ ta dừng sau **tối đa 9 lượt**, trước khi `x` về `0`. (Nếu `n >= 55`, lượt thứ `11` phải bốc `0` viên và `n >= 0` luôn đúng — vòng lặp sẽ không dừng; bài này loại trừ bằng `n <= 50`.)

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1 — mô phỏng có `turn` | `O(sqrt(n))` | `O(1)` | gần đề nhất |
| 2 — đếm `k` rồi xét `k % 2` | `O(sqrt(n))` | `O(1)` | lời giải chính thức |
| 3 — đệ quy `win(n, x)` | `O(sqrt(n))` | `O(sqrt(n))` | mẫu cho trò chơi tổng quát |
| 4 — `accumulate` + `bisect_right` | `O(log m)` / truy vấn | `O(m)` | `m = 10` mốc |
| 5 — bảng khoảng | `O(1)` | `O(1)` | chỉ đúng `n <= 50`, phải kiểm chứng |

Số lượt tối đa là `9` (vì `10 + ... + 2 = 54 > 50`), tức mọi cách đều **hằng số** theo ràng buộc.

## 12. Bộ test

Gom các cách vào `impls`. Oracle độc lập: **mô phỏng bằng danh sách lượt** — lấy lần lượt phần tử của `[10, 9, ..., 1]`, trừ dần, và xem ai đi lượt cuối; viết khác với các cách trên để làm đối chứng.

In [21]:
def oracle(n: int) -> bool:
    """Mô phỏng bằng danh sách lượt cố định; trả True nếu Alice thắng."""
    last_mover = None                 # None: chưa ai đi được
    pile = n
    for turn_index, need in enumerate(range(10, 0, -1)):     # need = 10, 9, ..., 1
        if pile < need:
            break
        pile -= need
        last_mover = "Alice" if turn_index % 2 == 0 else "Bob"
    return last_mover == "Alice"

impls = {
    "turns": can_alice_win_turns,
    "count": can_alice_win_count,
    "recursive": can_alice_win_recursive,
    "bisect": can_alice_win_bisect,
    "ranges": can_alice_win_ranges,
    "Solution": Solution().canAliceWin,
    "SolutionCount": SolutionCount().canAliceWin,
    "SolutionRecursive": SolutionRecursive().canAliceWin,
    "SolutionBisect": SolutionBisect().canAliceWin,
    "SolutionRanges": SolutionRanges().canAliceWin,
    "oracle": oracle,
}

def check_all(n, expected, note=""):
    for name, f in impls.items():
        got = f(n)
        assert got == expected, (name, n, got, expected)
    print(f"OK  n={n} -> {expected}  {note}")

check_all(12, True, "ví dụ 1")
check_all(1, False, "ví dụ 2")

OK  n=12 -> True  ví dụ 1
OK  n=1 -> False  ví dụ 2


### 12.1. Các trường hợp biên

In [22]:
check_all(9, False, "ngay dưới 10: Alice không bốc được")
check_all(10, True, "đúng 10: Alice bốc hết, Bob không bốc được 9")
check_all(18, True, "Alice 10, còn 8 < 9: Bob thua")
check_all(19, False, "Alice 10, Bob 9, còn 0: Alice không bốc 8 được")
check_all(26, False, "10+9 = 19, còn 7 < 8 nên k = 2 (Alice không bốc được 8)")
check_all(27, True, "10 + 9 + 8 = 27: k = 3, Bob không bốc được 7")
check_all(33, True, "k = 3, còn 6 < 7")
check_all(34, False, "10+9+8+7 = 34: k = 4")
check_all(44, True, "k = 5, còn 4 < 5")
check_all(45, False, "10+9+8+7+6+5 = 45: k = 6")
check_all(49, True, "10+...+4 = 49: k = 7")
check_all(50, True, "biên trên: k = 7, còn 1")
check_all(40, True, "10+9+8+7+6 = 40: k = 5")

OK  n=9 -> False  ngay dưới 10: Alice không bốc được
OK  n=10 -> True  đúng 10: Alice bốc hết, Bob không bốc được 9
OK  n=18 -> True  Alice 10, còn 8 < 9: Bob thua
OK  n=19 -> False  Alice 10, Bob 9, còn 0: Alice không bốc 8 được
OK  n=26 -> False  10+9 = 19, còn 7 < 8 nên k = 2 (Alice không bốc được 8)
OK  n=27 -> True  10 + 9 + 8 = 27: k = 3, Bob không bốc được 7
OK  n=33 -> True  k = 3, còn 6 < 7
OK  n=34 -> False  10+9+8+7 = 34: k = 4
OK  n=44 -> True  k = 5, còn 4 < 5
OK  n=45 -> False  10+9+8+7+6+5 = 45: k = 6
OK  n=49 -> True  10+...+4 = 49: k = 7
OK  n=50 -> True  biên trên: k = 7, còn 1
OK  n=40 -> True  10+9+8+7+6 = 40: k = 5


### 12.2. Phản ví dụ cho các cách làm sai

Ba cách "gần đúng": (a) dùng `n > x` (không cho bốc hết đống); (b) đảo điều kiện thắng (`k % 2 == 0`); (c) chỉ xét chẵn/lẻ của `n`. Ta tìm các `n` làm chúng sai so với oracle.

In [23]:
def wrong_strict(n):
    x, k = 10, 0
    while n > x:               # SAI: phải là >=
        n -= x
        x -= 1
        k += 1
    return k % 2 == 1

def wrong_flipped(n):
    x, k = 10, 0
    while n >= x:
        n -= x
        x -= 1
        k += 1
    return k % 2 == 0          # SAI: đảo điều kiện

def wrong_parity_of_n(n):
    return n % 2 == 0          # SAI: không liên quan tới số lượt

for w in (wrong_strict, wrong_flipped, wrong_parity_of_n):
    bad = [n for n in range(1, 51) if w(n) != oracle(n)]
    print(f"{w.__name__:<18} sai ở {len(bad):>2} giá trị n, ví dụ {bad[:6]}")
    assert bad

wrong_strict       sai ở  7 giá trị n, ví dụ [10, 19, 27, 34, 40, 45]
wrong_flipped      sai ở 50 giá trị n, ví dụ [1, 2, 3, 4, 5, 6]
wrong_parity_of_n  sai ở 24 giá trị n, ví dụ [2, 4, 6, 8, 11, 13]


### 12.3. Đối chiếu toàn bộ `n = 1..50` và mở rộng

Miền hợp lệ chỉ có `50` giá trị nên ta **đối chiếu tất cả**. Ta cũng kiểm tra một **tính chất**: các `n` mà Alice thắng chính là những `n` nằm giữa mốc thứ `k` và `k + 1` với `k` lẻ. Cuối cùng chạy một bản "an toàn" `can_alice_win_safe` có điều kiện `x > 0` cho `n` tới `500` để thấy vòng lặp **dừng** (ngoài ràng buộc, chỉ để tham khảo).

In [24]:
for n in range(1, 51):
    results = {name: f(n) for name, f in impls.items()}
    assert len(set(results.values())) == 1, (n, results)
print("Đối chiếu đủ 50 giá trị: OK")

# Tính chất: Alice thắng <=> số mốc 'prefix' không vượt quá n là số lẻ
for n in range(1, 51):
    k = sum(1 for p in PREFIX if p <= n)
    assert (k % 2 == 1) == Solution().canAliceWin(n)
print("Tính chất số mốc lẻ: OK")

def can_alice_win_safe(n: int) -> bool:
    x, k = 10, 0
    while x > 0 and n >= x:        # dừng khi x chạm 0 (ngoài ràng buộc)
        n -= x
        x -= 1
        k += 1
    return k % 2 == 1

for n in range(1, 51):
    assert can_alice_win_safe(n) == Solution().canAliceWin(n)
print("Bản an toàn (có x > 0) khớp trên 1..50; n = 55 ->", can_alice_win_safe(55), ", n = 500 ->", can_alice_win_safe(500))

Đối chiếu đủ 50 giá trị: OK
Tính chất số mốc lẻ: OK
Bản an toàn (có x > 0) khớp trên 1..50; n = 55 -> False , n = 500 -> False


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `n > x` thay vì `n >= x`

Nếu đống đúng bằng `x` viên thì người chơi **vẫn bốc được** (bốc hết). `n > x` coi như không bốc được.

In [25]:
print(wrong_strict(10), "(sai) vs", Solution().canAliceWin(10), "(đúng)")
assert wrong_strict(10) is False and Solution().canAliceWin(10) is True

False (sai) vs True (đúng)


**Vì sao sai:** ví dụ `n = 10`: Alice bốc đúng `10` viên (hết đống), rồi Bob không bốc được `9` ⇒ Alice thắng. `n > x` bỏ sót lượt đó. **Sửa:** `while n >= x`.

### Lỗi 2: Đảo điều kiện thắng (`k % 2 == 0`)

In [26]:
print(wrong_flipped(12), "(sai) vs", Solution().canAliceWin(12), "(đúng)")
assert wrong_flipped(12) is False and Solution().canAliceWin(12) is True

False (sai) vs True (đúng)


**Vì sao sai:** `k` là số lượt **đã đi được**; người đi **lượt cuối** (lượt thứ `k`) mới là người **thắng**. Alice đi lượt lẻ ⇒ thắng khi `k` lẻ. Với `n = 12`, `k = 1` (Alice bốc `10`), nên `k % 2 == 1`.

### Lỗi 3: Nghĩ rằng chỉ cần xem `n` chẵn hay lẻ

In [27]:
bad = [n for n in range(1, 51) if wrong_parity_of_n(n) != Solution().canAliceWin(n)]
print("n chẵn/lẻ không quyết định kết quả; sai ở", len(bad), "giá trị, ví dụ", bad[:8])
print([ (n, Solution().canAliceWin(n)) for n in (10, 11, 12, 13)])     # cả chẵn lẫn lẻ đều True

n chẵn/lẻ không quyết định kết quả; sai ở 24 giá trị, ví dụ [2, 4, 6, 8, 11, 13, 15, 17]
[(10, True), (11, True), (12, True), (13, True)]


**Vì sao sai:** kết quả phụ thuộc **số lượt đi được**, quyết định bởi các mốc `10, 19, 27, 34, ...` chứ không phải tính chẵn lẻ của `n`. Ví dụ `n = 10, 11, 12, 13` đều cho `True` dù chẵn hay lẻ.

### Lỗi 4: Quên điều kiện dừng khi `x` chạm `0` (ngoài ràng buộc)

Với `n >= 55`, sau `10` lượt `x = 0` và `n >= 0` luôn đúng, `x` thành âm và `n` còn **tăng** (`n -= x` với `x < 0`): vòng lặp không bao giờ dừng. Bài này an toàn nhờ `n <= 50`, nhưng viết thêm `x > 0` là thói quen tốt.

In [28]:
def sai_khong_chan_x(n, max_iter=15):
    """Giả lập vòng lặp không có x > 0; giới hạn số vòng để không treo máy."""
    x, k = 10, 0
    history = []
    while n >= x and k < max_iter:
        history.append((k, n, x))
        n -= x
        x -= 1
        k += 1
    return history

h = sai_khong_chan_x(55)
print(h[8:])        # sau lượt x = 1: x = 0 rồi x âm, n bắt đầu TĂNG lại
print("vòng lặp chưa dừng sau 15 vòng:", len(h) == 15)
assert len(h) == 15
assert can_alice_win_safe(55) in (True, False)    # bản có x > 0 thì dừng

[(8, 3, 2), (9, 1, 1), (10, 0, 0), (11, 0, -1), (12, 1, -2), (13, 3, -3), (14, 6, -4)]
vòng lặp chưa dừng sau 15 vòng: True


**Vì sao nguy hiểm:** vòng lặp chỉ dừng khi `n < x`. Khi `x <= 0`, điều kiện `n >= x` luôn đúng (với `n >= 0`) và `n` không bao giờ nhỏ hơn `x` ⇒ treo. **Sửa:** thêm `x > 0` vào điều kiện `while`, hoặc dựa chắc vào ràng buộc `n <= 50`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Trò chơi "không có lựa chọn"** (mỗi lượt đi cố định) **chỉ là mô phỏng** — đừng vội nghĩ tới minimax hay DP. Hãy hỏi trước: "mỗi người có được chọn không?".
- **"Ai đi lượt cuối thì thắng" → chẵn/lẻ của số lượt.** Quy tắc `k % 2` xuất hiện trong nhiều bài trò chơi (Nim, Divisor Game).
- **Chuỗi giảm dần `10, 9, 8, ...`**: tổng dồn `10, 19, 27, ...` cho phép trả lời bằng `bisect` hoặc bảng khoảng.
- **Ràng buộc nhỏ (`n <= 50`)** → có thể vét cạn **mọi** input để kiểm chứng công thức.
- **Điều kiện `>=` vs `>`** quyết định bài này: hãy đọc kỹ "không thể thực hiện lượt" nghĩa là gì (số viên còn lại **ít hơn** số cần bốc).
- **Game tổng quát (có lựa chọn):** dùng `win(state)` = tồn tại nước đi làm đối thủ thua. Bài này chỉ có một nước đi nên `win = not win(next)`.

## 15. Biến thể của bài

1. **Số viên khởi đầu khác** (`start` thay cho `10`): `canAliceWin(n, start)`.
2. **Bob đi trước** hoặc đếm số `n` mà Bob thắng.
3. **Giảm `d` mỗi lượt** thay vì `1` (`10, 8, 6, ...`), dừng khi `x <= 0`.
4. **Người chơi được chọn** bốc `x` hoặc `x - 1` viên (có lựa chọn) — cần minimax/DP thật sự.

In [29]:
def can_first_win(n: int, start: int = 10, dec: int = 1) -> bool:
    """Biến thể tổng quát: người đi trước bốc `start`, mỗi lượt sau bớt `dec`; dừng khi x <= 0."""
    x, k = start, 0
    while x > 0 and n >= x:
        n -= x
        x -= dec
        k += 1
    return k % 2 == 1

# Với start = 10, dec = 1 khớp bài gốc
for n in range(1, 51):
    assert can_first_win(n) == Solution().canAliceWin(n)

print(can_first_win(7, start=5))            # 5, rồi Bob cần 4 còn 2 -> k = 1 -> True
print(can_first_win(9, start=5))            # 5, rồi Bob 4, còn 0; Alice cần 3 -> k = 2 -> False
print([n for n in range(1, 21) if can_first_win(n, start=10, dec=2)])   # giảm 2 mỗi lượt: 10, 8, 6, 4, 2

def count_alice_wins(limit: int = 50) -> int:
    return sum(1 for n in range(1, limit + 1) if Solution().canAliceWin(n))

print(count_alice_wins())    # số n trong 1..50 mà Alice thắng

True
False
[10, 11, 12, 13, 14, 15, 16, 17]
23


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Mô phỏng có biến `turn`, ai không bốc được thì thua | `O(sqrt(n))` | `O(1)` | bám sát đề, mới học |
| 2 | Đếm số lượt `k`, Alice thắng khi `k` lẻ | `O(sqrt(n))` | `O(1)` | lời giải chuẩn |
| 3 | Đệ quy `win(n, x) = not win(n - x, x - 1)` | `O(sqrt(n))` | `O(sqrt(n))` | mẫu cho trò chơi tổng quát |
| 4 | `accumulate` + `bisect_right` | `O(log m)` / truy vấn | `O(m)` | nhiều truy vấn `n` |
| 5 | Bảng khoảng cứng | `O(1)` | `O(1)` | chỉ khi miền cố định, phải kiểm chứng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `Solution().canAliceWin(10)`, `Solution().canAliceWin(9)`, `Solution().canAliceWin(19)`, `Solution().canAliceWin(27)`, `list(accumulate([10, 9, 8]))`.

**Bài 2 (tính tay).** Với `n = 40` và `n = 26`: lập bảng từng lượt (người đi, `x`, `n` trước, `n` sau), tìm `k` và người thắng.

**Bài 3 (điền TODO).** Hoàn thành `can_alice_win_todo(n)` theo Cách 2.

In [30]:
def can_alice_win_todo(n):
    # TODO 1: x = 10 (số viên phải bốc), k = 0 (số lượt đã đi)
    # TODO 2: while n >= x: trừ x khỏi n, giảm x đi 1, tăng k
    # TODO 3: trả về True nếu k lẻ
    pass

# Tự làm xong rồi mới mở test:
# assert can_alice_win_todo(12) is True
# assert can_alice_win_todo(1) is False

**Bài 4 (tự viết lại).** Viết `can_alice_win_v2(n)` dùng `for need in range(10, 0, -1)` và `break` (không dùng `while`): đếm số lượt `k` rồi trả `k % 2 == 1`. So sánh với `Solution().canAliceWin` cho mọi `n = 1..50`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `winning_intervals(limit)` trả về danh sách các khoảng `(lo, hi)` liên tiếp (`1 <= lo <= hi <= limit`) mà Alice thắng; kiểm tra `winning_intervals(50) == [(10, 18), (27, 33), (40, 44), (49, 50)]`. (b) Với biến thể "Alice bốc `start` viên đầu tiên, mỗi lượt giảm `1`" (dừng khi `x = 0`), tìm `n` nhỏ nhất `>= 1` mà Alice thắng khi `start = 4`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `True` (10 bốc hết, Bob không bốc 9), `False` (`9 < 10`), `False` (`k = 2`), `True` (`k = 3`), `[10, 19, 27]`. Bài 3: ba dòng chính: khởi tạo, `while n >= x`, trả `k % 2 == 1`.
- **Mức 2 (vừa):** Bài 2: `n = 40`: `10, 9, 8, 7, 6` đúng `40`, `k = 5` lẻ ⇒ Alice thắng (Bob cần `5` nhưng còn `0`). `n = 26`: `10, 9` ⇒ còn `7`; Alice cần `8` ⇒ không đi được; `k = 2` ⇒ Bob thắng. Bài 4: `k += 1` mỗi lần trừ được; `break` khi `n < need`.
- **Mức 3 (sâu):** Bài 5(a): duyệt `n = 1..limit`, gom các `n` liên tiếp có `canAliceWin` là `True` thành khoảng. 5(b): với `start = 4` các mốc là `4, 7, 9, 10`; hãy liệt kê `k` theo `n` và chọn `n` nhỏ nhất có `k` lẻ (thử bằng `can_first_win`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert Solution().canAliceWin(10) is True
assert Solution().canAliceWin(9) is False
assert Solution().canAliceWin(19) is False
assert Solution().canAliceWin(27) is True
assert list(accumulate([10, 9, 8])) == [10, 19, 27]

# Bài 2
assert count_moves(40) == 5 and Solution().canAliceWin(40) is True     # 10+9+8+7+6 = 40
assert count_moves(26) == 2 and Solution().canAliceWin(26) is False    # 10 + 9 = 19, còn 7 < 8

# Bài 3
def can_alice_win_todo_solution(n):
    x, k = 10, 0
    while n >= x:
        n -= x
        x -= 1
        k += 1
    return k % 2 == 1

assert can_alice_win_todo_solution(12) is True
assert can_alice_win_todo_solution(1) is False

# Bài 4
def can_alice_win_v2(n):
    k = 0
    for need in range(10, 0, -1):
        if n < need:
            break
        n -= need
        k += 1
    return k % 2 == 1

for n in range(1, 51):
    assert can_alice_win_v2(n) == Solution().canAliceWin(n)

# Bài 5(a)
def winning_intervals(limit):
    res = []
    start = None
    for n in range(1, limit + 1):
        if Solution().canAliceWin(n):
            if start is None:
                start = n
        else:
            if start is not None:
                res.append((start, n - 1))
                start = None
    if start is not None:
        res.append((start, limit))
    return res

assert winning_intervals(50) == [(10, 18), (27, 33), (40, 44), (49, 50)]

# Bài 5(b): start = 4, mốc 4, 7, 9, 10 -> k lẻ ở [4, 6] (k = 1)
smallest = next(n for n in range(1, 100) if can_first_win(n, start=4))
assert smallest == 4
assert [n for n in range(1, 13) if can_first_win(n, start=4)] == [4, 5, 6, 9]
print("Tất cả đáp án đều đúng.")

Tất cả đáp án đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (mô phỏng, đếm số lượt, xét chẵn/lẻ)

```python
class Solution:
    def canAliceWin(self, n: int) -> bool:
        x, k = 10, 0
        while n >= x:
            n -= x
            x -= 1
            k += 1
        return k % 2 == 1
```

**Checklist**

- [ ] Điều kiện lặp là `n >= x` (đống đúng bằng `x` vẫn bốc được)?
- [ ] `x` khởi đầu `10` và **giảm `1`** sau mỗi lượt; `k` tăng mỗi lượt?
- [ ] Alice thắng khi `k` **lẻ** (Alice đi lượt `1, 3, 5, ...`)?
- [ ] `n < 10` trả `False` ngay (`k = 0`)?
- [ ] Biết vì sao vòng lặp dừng (`10 + ... + 1 = 55 > 50`), hoặc đã thêm `x > 0`?
- [ ] Đã thử `n = 1 -> False`, `10 -> True`, `12 -> True`, `19 -> False`, `27 -> True`, `50 -> True`?
- [ ] Skeleton đúng `class Solution` / `def canAliceWin(self, n: int) -> bool`?